In [1]:
import duckdb

In [2]:
conn = duckdb.connect("ai_shelf.duckdb")

In [8]:
import os
print(os.listdir("../data/raw/amazon"))

['.gitkeep', 'Beauty_and_Personal_Care.jsonl', 'meta_Beauty_and_Personal_Care.jsonl']


In [11]:
# Caminhos dos arquivos de dados, relativos à raiz do projeto (AI-SHELF/)
REVIEWS_PATH = "../data/raw/amazon/Beauty_and_Personal_Care.jsonl"
META_PATH = "../data/raw/amazon/meta_Beauty_and_Personal_Care.jsonl"

In [12]:
# Schema das reviews — read_json_auto infere as colunas direto do .jsonl.gz,
# sem precisar descompactar ou declarar CREATE TABLE manualmente.
# Checar se existem: rating, text, asin/parent_asin, timestamp
print("=== SCHEMA: REVIEWS ===")
conn.sql(f"DESCRIBE SELECT * FROM read_json_auto('{REVIEWS_PATH}')").show()

=== SCHEMA: REVIEWS ===
┌───────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │                                                  column_type                                                  │  null   │   key   │ default │  extra  │
│      varchar      │                                                    varchar                                                    │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ rating            │ DOUBLE                                                                                                        │ YES     │ NULL    │ NULL    │ NULL    │
│ title             │ VARCHAR                                                                             

In [13]:
# Schema do metadata dos produtos
# Checar se existem: average_rating, price, store, categories
print("=== SCHEMA: META ===")
conn.sql(f"DESCRIBE SELECT * FROM read_json_auto('{META_PATH}')").show()

=== SCHEMA: META ===
┌─────────────────┬───────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │                                column_type                                │  null   │   key   │ default │  extra  │
│     varchar     │                                  varchar                                  │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼───────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ main_category   │ VARCHAR                                                                   │ YES     │ NULL    │ NULL    │ NULL    │
│ title           │ VARCHAR                                                                   │ YES     │ NULL    │ NULL    │ NULL    │
│ average_rating  │ DOUBLE                                                                    │ YES     │ NULL    │ NULL    │ NULL    │
│ rating_number   │ BIGINT 

In [14]:
# Amostra de linhas reais — schema mostra o tipo, mas não mostra se o
# conteúdo é útil (texto vazio, encoding quebrado, campos nulos em excesso)
print("=== SAMPLE: REVIEWS (5 linhas) ===")
conn.sql(f"SELECT * FROM read_json_auto('{REVIEWS_PATH}') LIMIT 5").show()

=== SAMPLE: REVIEWS (5 linhas) ===
┌────────┬───────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [15]:
print("=== SAMPLE: META (5 linhas) ===")
conn.sql(f"SELECT * FROM read_json_auto('{META_PATH}') LIMIT 5").show()

=== SAMPLE: META (5 linhas) ===
┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────┬───────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [16]:
# Contagem real de linhas — o site oficial informa 23.9M reviews e 1.0M
# itens para esta categoria. Confirmar aqui, não confiar cegamente no número.
# Aviso: obriga o DuckDB a descomprimir e parsear o arquivo inteiro,
# pode demorar alguns minutos. Não é erro, é esperado dado o volume.
print("=== ROW COUNT: REVIEWS (pode demorar) ===")
conn.sql(f"SELECT count(*) AS total_reviews FROM read_json_auto('{REVIEWS_PATH}')").show()

=== ROW COUNT: REVIEWS (pode demorar) ===
┌───────────────┐
│ total_reviews │
│     int64     │
├───────────────┤
│      23911390 │
└───────────────┘



In [17]:
print("=== ROW COUNT: META ===")
conn.sql(f"SELECT count(*) AS total_products FROM read_json_auto('{META_PATH}')").show()

=== ROW COUNT: META ===
┌────────────────┐
│ total_products │
│     int64      │
├────────────────┤
│        1028914 │
└────────────────┘



In [19]:
# Checar quantos produtos têm price preenchido vs. nulo
# Se a taxa de NULL for muito alta (ex: >40-50%), price não pode entrar
# como feature direto sem uma estratégia de imputação ou fallback
print("=== NULL RATE: price, average_rating, store (META) ===")
conn.sql(f"""
    SELECT
        count(*) AS total,
        sum(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS price_null,
        round(100.0 * sum(CASE WHEN price IS NULL THEN 1 ELSE 0 END) / count(*), 1) AS price_null_pct,
        sum(CASE WHEN store IS NULL THEN 1 ELSE 0 END) AS store_null,
        sum(CASE WHEN average_rating IS NULL THEN 1 ELSE 0 END) AS rating_null
    FROM read_json_auto('{META_PATH}', sample_size=-1)
""").show()

=== NULL RATE: price, average_rating, store (META) ===
┌─────────┬────────────┬────────────────┬────────────┬─────────────┐
│  total  │ price_null │ price_null_pct │ store_null │ rating_null │
│  int64  │   int128   │     double     │   int128   │   int128    │
├─────────┼────────────┼────────────────┼────────────┼─────────────┤
│ 1028914 │     648234 │           63.0 │      50710 │           0 │
└─────────┴────────────┴────────────────┴────────────┴─────────────┘



In [20]:
print("=== PRICE FORMAT CHECK: quantos non-null NÃO são número puro ===")
conn.sql(f"""
    SELECT
        count(*) AS total_non_null,
        sum(CASE WHEN try_cast(price AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numerico,
        sum(CASE WHEN try_cast(price AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS numerico_limpo
    FROM read_json_auto('{META_PATH}', sample_size=-1)
    WHERE price IS NOT NULL
""").show()

=== PRICE FORMAT CHECK: quantos non-null NÃO são número puro ===
┌────────────────┬──────────────┬────────────────┐
│ total_non_null │ nao_numerico │ numerico_limpo │
│     int64      │    int128    │     int128     │
├────────────────┼──────────────┼────────────────┤
│         380680 │           83 │         380597 │
└────────────────┴──────────────┴────────────────┘



In [21]:
# parent_asin duplicado quebraria o JOIN produto x review (contaria review 2x)
print("=== DUPLICATE CHECK: parent_asin (META) ===")
conn.sql(f"""
    SELECT parent_asin, count(*) AS n
    FROM read_json_auto('{META_PATH}', sample_size=-1)
    GROUP BY parent_asin
    HAVING count(*) > 1
    ORDER BY n DESC
    LIMIT 10
""").show()

=== DUPLICATE CHECK: parent_asin (META) ===
┌─────────────┬───────┐
│ parent_asin │   n   │
│   varchar   │ int64 │
└─────────────┴───────┘
        0 rows       



In [22]:
# text vazio = sem uso pro NLP (Fase 7/8) | rating nulo = sem uso pro ranking
print("=== NULL RATE: text, rating (REVIEWS) ===")
conn.sql(f"""
    SELECT
        count(*) AS total,
        sum(CASE WHEN text IS NULL OR trim(text) = '' THEN 1 ELSE 0 END) AS text_vazio,
        round(100.0 * sum(CASE WHEN text IS NULL OR trim(text) = '' THEN 1 ELSE 0 END) / count(*), 1) AS text_vazio_pct,
        sum(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS rating_null
    FROM read_json_auto('{REVIEWS_PATH}')
""").show()

=== NULL RATE: text, rating (REVIEWS) ===
┌──────────┬────────────┬────────────────┬─────────────┐
│  total   │ text_vazio │ text_vazio_pct │ rating_null │
│  int64   │   int128   │     double     │   int128    │
├──────────┼────────────┼────────────────┼─────────────┤
│ 23911390 │      36594 │            0.2 │           0 │
└──────────┴────────────┴────────────────┴─────────────┘



In [24]:
print("=== price NULL vs rating_number (é produto pouco avaliado?) ===")
conn.sql(f"""
    SELECT
        CASE WHEN price IS NULL THEN 'sem_price' ELSE 'com_price' END AS grupo,
        avg(rating_number) AS media_reviews,
        avg(average_rating) AS media_rating
    FROM read_json_auto('{META_PATH}', sample_size=-1)
    GROUP BY grupo
""").show()

=== price NULL vs rating_number (é produto pouco avaliado?) ===
┌───────────┬────────────────────┬──────────────────┐
│   grupo   │   media_reviews    │   media_rating   │
│  varchar  │       double       │      double      │
├───────────┼────────────────────┼──────────────────┤
│ sem_price │ 100.05310736555009 │  3.9883299857768 │
│ com_price │  436.5767153514763 │ 4.18058763265736 │
└───────────┴────────────────────┴──────────────────┘



In [25]:
# Roda os dois SQLs, criando as tabelas de verdade dentro do ai_shelf.duckdb
conn.sql(open("../sql/02_products.sql").read())
conn.sql(open("../sql/03_reviews.sql").read())

# Confere se bateu com os números do Data Audit (1.028.914 / 23.911.390)
# Se não bater, algum SELECT filtrou ou duplicou linha sem querer
conn.sql("SELECT count(*) AS total_products FROM products").show()
conn.sql("SELECT count(*) AS total_reviews FROM reviews").show()

┌────────────────┐
│ total_products │
│     int64      │
├────────────────┤
│        1028914 │
└────────────────┘

┌───────────────┐
│ total_reviews │
│     int64     │
├───────────────┤
│      23911390 │
└───────────────┘



In [26]:
conn.sql("SELECT * FROM products LIMIT 5").show()

┌─────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────┬───────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────┬───────────────┬────────┬───────────┐
│ parent_asin │                                                                                                               title                                                                                                               │   store    │ main_category │                                                  categories                                                  │ average_rating │ rating_number │ price  │ has_price │
│   varchar   │                                                                                                             

In [27]:
conn.sql("SELECT * FROM reviews LIMIT 5").show()

┌─────────────┬────────────┬──────────────────────────────┬────────┬───────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [28]:
conn.close()